# NB12 — S4b: confirmation on two additional architectures

18 NEW jobs: ConvNeXt-V2 Tiny and MobileNetV4 × 3 factors × seeds 1/2/3,
fold 1 only, 60 epochs each. Six existing Stage-A baselines are reused.
No annotation work, no NB11, no SAM2, no change to existing models.

Factors are the three highest signed mean effects in the original NB06 report:
class-weighted sampling, random initialisation, uniform sampling. Only the first
was positive in discovery. Confirm the directions honestly, not three claimed gains.
Selection uses the frozen HF revision, never the new confirmation results.
This is a declared extension because NB06 already covered three architectures;
it is not a blind test on new data or proof of significance.

Run instructions: Internet ON; HF_TOKEN secret; attach Tire Dataset Prepared;
GPU T4 x2. Run All. For FOUR accounts, set the same ACTIVE_KAGGLE_ACCOUNTS
tuple to ('acct1','acct2','acct3','acct4') in every copy and set ACCOUNT per copy.
Default is one copy. Run all copies with this identical notebook/protocol.
Do not change model, batch, seed, epoch count or factor settings.

Models run in isolated child processes to release GPU/host memory. Scratch and
checkpoints use /kaggle/temp; monitor actual disk, do not assume 1 TB is available.
HF batches ordinary results every 30 minutes, plus major completion and catchable
Stop; work-takeover coordination can require a separate small commit. Hard OS
kills cannot flush. Fresh sessions resume the last published COMPLETE EPOCH,
not the exact interrupted batch. Strict resume refuses corrupt/mismatched checkpoints.
Do not run NB12R until all 18 jobs are FINISHED. GPU preflight runs before claims.


In [1]:
# === CELL 1 of every notebook: unpack the library ==========================
# Writes tyrelib.py into the session and imports it. Nothing here touches the
# GPU or the network beyond installing three small packages.
#
#   tyrelib   the whole pipeline: HuggingFace sync, registry, work sharding,
#             telemetry, model zoo, training loop, metrics.
#
# Generated by build_notebooks.py from tyrelib.py. Editing the blob below does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle ships torch, pandas, sklearn. These vary by image version, so check.
#   pynvml  reads GPU power/temperature/clocks directly (per device)
#   psutil  peak RAM and CPU
#   pyarrow writes per-sample predictions as Parquet
for _pkg in ('pynvml', 'psutil', 'pyarrow', 'timm'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                       check=False)

_LIB = (
    '...',
)

(WORK / 'tyrelib.py').write_bytes(base64.b64decode(''.join(_LIB)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
# Without this, re-running cell 1 after an edit returns the cached module and
# you spend an hour debugging a ghost.
for _m in [m for m in list(sys.modules) if m == 'tyrelib']:
    del sys.modules[_m]
import tyrelib as tl
print('tyrelib', tl.__version__, 'loaded')


tyrelib v12 loaded


## Session — Internet ON, HF_TOKEN secret; one notebook by default

In [2]:
# === Who am I? =============================================================
#
# ACCOUNT labels this Kaggle account in the shared run log.
# ACTIVE_KAGGLE_ACCOUNTS is the ONE source of truth for parallelism.
# NUM_WORKERS and WORKER_ID are derived from it; do not edit them.
#
# ---------------------------------------------------------------------------
# THESE TWO VALUES ARE SAFE TO CHANGE AT ANY TIME.
#
# They decide which account owns each fresh run. An absent or partial run stays
# with that static owner. Work stealing is disabled by default; an explicit
# recovery run may opt in and can then take over only a real claim/run event
# older than 45 minutes. Whether a run is finished, and what epoch it reached,
# is read from HuggingFace -- from the run's own files -- so it is the same
# answer for every account at every worker count.
# Go from 4 workers to 1 and nothing is retrained: the runs the other three
# finished are skipped, and the ones they left half-done are RESUMED from
# their checkpoints.
#
# (It did not always work that way. Resume used to check only the local disk,
#  and Kaggle wipes that between sessions, so every run restarted at epoch 1.
#  See docs/05 -- Bug 8.)
# ---------------------------------------------------------------------------
#
# All accounts push to the SAME HuggingFace account (Shanmuk4622), so the
# 128-writes-per-hour budget is SHARED. tyrelib caps each worker at
# 100/NUM_WORKERS automatically.
#
# DEFAULT: exactly one Kaggle notebook. For four parallel copies, replace the
# tuple with ('acct1', 'acct2', 'acct3', 'acct4') in every copy and set ACCOUNT
# to that copy's label. Keeping the active labels in one tuple prevents a cell
# that says "one worker" in one place but silently launches as worker 0/4.
ACTIVE_KAGGLE_ACCOUNTS = ('acct1', 'acct2', 'acct3', 'acct4')   # <<< one notebook; list all four only when all four run
ACCOUNT = 'acct2'                     # <<< this copy's label

# A missing comma in ('acct1') makes it a string. tyrelib deliberately repairs
# that common edit, so a valid one-worker session cannot fail before HF sync.
ACTIVE_KAGGLE_ACCOUNTS = tl.normalise_active_accounts(ACTIVE_KAGGLE_ACCOUNTS)
if ACCOUNT not in ACTIVE_KAGGLE_ACCOUNTS:
    raise ValueError(f"ACCOUNT={ACCOUNT!r} is not active: {ACTIVE_KAGGLE_ACCOUNTS}")
NUM_WORKERS = len(ACTIVE_KAGGLE_ACCOUNTS)
WORKER_ID = ACTIVE_KAGGLE_ACCOUNTS.index(ACCOUNT)
print(f"RUN MODE CHECK: {ACCOUNT=} {ACTIVE_KAGGLE_ACCOUNTS=} -> worker {WORKER_ID}/{NUM_WORKERS}")

sess = tl.Session(account=ACCOUNT, worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                  stage='c',
                  hf_repo='Shanmuk4622/tyre-wear-study',
                  enable_hf=True,
                  session_limit_h=8.5,      # push + pause before Kaggle kills us
                  push_interval_min=30)     # background commit cycle


RUN MODE CHECK: ACCOUNT='acct2' ACTIVE_KAGGLE_ACCOUNTS=('acct1', 'acct2', 'acct3', 'acct4') -> worker 1/4
[DISK] staging /kaggle/temp/tyre_study  (1102 GB free)
[HF] authenticated as Shanmuk4622  ->  dataset:Shanmuk4622/tyre-wear-study
[HF] rate cap 25/hr (shared across all workers on this token)
[HF] background uploader started (30 min cycle)
[LIFE] guards installed (SIGTERM, SIGINT, atexit, watchdog @ 8.5 h)

[SESSION] account=acct2  worker=1/4  stage=c  id=d646f0
[SESSION] MODE=4 PARALLEL NOTEBOOKS: each account starts with one static shard, then safely helps when idle
[SESSION] staging /kaggle/temp/tyre_study  |  hf ON  |  cap 25/hr  |  push every 30 min
[SESSION] NUM_WORKERS assigns each FRESH run to one static owner. Completed/resumable state still comes from HuggingFace.



In [3]:
# === Find the dataset ======================================================
# One Kaggle dataset holds the whole package:
#     <slug>/FINAL/{images,splits,manifests}
#     <slug>/annotations/{clean,propagated}
# Kaggle sometimes wraps uploads in one more directory, so both are searched for.
DATA_ROOT = sess.prepare_data()
ANN_ROOT  = tl.find_annotations_root(DATA_ROOT)
print("annotations:", ANN_ROOT if ANN_ROOT else "NOT FOUND (only needed from NB08 onward)")


[DATA] root /kaggle/input/datasets/shanmuk4622/tire-dataset-prepared/Tire Dataset Prepared/FINAL
[DATA] 418 clean / 4180 derivatives / 12 sessions
annotations: /kaggle/input/datasets/shanmuk4622/tire-dataset-prepared/Tire Dataset Prepared/annotations


In [4]:
"""Locked, descriptive S4b protocol. Does not train or access the network."""
import hashlib
import json
import numpy as np
import pandas as pd

SOURCE_REVISION = 'bf62f9e9cbedacc580aa42542da14a068b8f9215'
REVISION = 's4b-2026-09-09-r1'
FACTORS = {
    'sampler_classweighted': {'sampler_name':'class_weighted'},
    'transfer_random': {'pretrained':False},
    'sampler_uniform': {'sampler_name':'uniform'},
}

def make_plan(selection, effects):
    assert selection.arch.is_unique
    assert set(selection.selection_revision) == {'2026-08-30-r3'}
    selected = selection.loc[selection.selected_top3.eq(True), 'arch'].tolist()
    assert set(selected)=={'regnety016','densenet121','resnet50'}
    eligible = selection[selection.eligible.eq(True) & selection.seeds.eq(3) & selection.xai_status.eq('ok')]
    targets = eligible[~eligible.arch.isin(selected)].sort_values(['ter_norm','bar','arch'],ascending=[False,True,True]).arch.tolist()
    assert targets==['convnextv2_t','mobilenetv4']
    assert len(effects)==108 and not effects.duplicated(['arch','factor','fold','seed']).any()
    assert set(effects.arch)==set(selected) and set(effects.fold)=={1}
    assert effects.groupby('factor').size().eq(9).all()
    assert set(effects.seed)=={1,2,3} and np.isfinite(effects.delta_vs_stage_a).all()
    ranking = effects.groupby('factor',as_index=False).delta_vs_stage_a.mean().sort_values(['delta_vs_stage_a','factor'],ascending=[False,True])
    factors = ranking.head(3).factor.tolist()
    assert factors==list(FACTORS), 'Frozen discovery ranking does not match this revision'
    return dict(revision=REVISION,source_revision=SOURCE_REVISION,architectures=targets,
        discovery_architectures=selected,factors=factors,overrides=FACTORS,folds=[1],seeds=[1,2,3],
        epochs=60,primary='best_val_f1_macro selected by best QWK, paired by architecture/fold/seed',
        secondary='final_val_f1_macro at epoch 60',
        ranking='descending mean signed paired NB06 selected-epoch F1 delta across all 9 discovery runs; alphabetical tie break',
        discovery_mean_delta=dict(zip(ranking.factor,ranking.delta_vs_stage_a)),
        claim='cross-architecture directional replication on the same fold; not an independent dataset or significance test')

def plan_hash(plan):
    return hashlib.sha256(json.dumps(plan,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def configs(sess, tl, plan):
    out=[]
    for factor in plan['factors']:
        for arch in plan['architectures']:
            for seed in plan['seeds']:
                cfg=sess.config(arch,1,seed,stage='c',technique='s4b_r1_'+factor,
                                _strict_resume=True,**FACTORS[factor])
                tl.validate_config(cfg)
                assert cfg['max_epochs']==60 and cfg['input_resolution']==384
                out.append(cfg)
    assert len(out)==18 and len({c['run_id'] for c in out})==18
    return out

def analyse(plan, trained, baseline):
    base=baseline.set_index(['arch','fold','seed'])
    assert base.index.is_unique and len(base)==6
    rows=[]
    for r in trained.to_dict('records'):
        key=(r['arch'],int(r['fold']),int(r['seed']))
        factor=r['technique'].removeprefix('s4b_r1_')
        assert factor in plan['factors']
        a=base.loc[key]
        rows.append(dict(run_id=r['run_id'],arch=r['arch'],factor=factor,fold=1,seed=key[2],
            selected_delta=float(r['best_val_f1_macro']-a.best_val_f1_macro),
            final_delta=float(r['final_val_f1_macro']-a.final_val_f1_macro)))
    paired=pd.DataFrame(rows)
    assert len(paired)==18 and not paired.duplicated(['arch','factor','seed']).any()
    assert np.isfinite(paired[['selected_delta','final_delta']]).all().all()
    summary=paired.groupby(['arch','factor']).agg(n=('seed','size'),mean_selected_delta=('selected_delta','mean'),
        sd_selected_delta=('selected_delta','std'),mean_final_delta=('final_delta','mean')).reset_index()
    assert summary.n.eq(3).all()
    summary['discovery_delta']=summary.factor.map(plan['discovery_mean_delta'])
    summary['same_direction']=summary.mean_selected_delta * summary.discovery_delta > 0
    summary['interpretation']=np.where(summary.same_direction,'same_direction_descriptive','not_replicated_direction')
    return paired,summary


In [5]:
from pathlib import Path
import json, numpy as np, pandas as pd
import re, time
from email.utils import parsedate_to_datetime
from huggingface_hub import HfApi, hf_hub_download
REPO = tl.HF_REPO_DEFAULT
READ_TOKEN = getattr(sess.uploader, "token", None)
if not READ_TOKEN:
    raise RuntimeError("HF_TOKEN is required: enable the Kaggle secret and rerun the Session cell.")

def hf_read(label, operation, attempts=8):
    # API metadata calls need retries too; uploads retain their existing schedule.
    for attempt in range(attempts):
        try:
            return operation()
        except Exception as exc:
            response = getattr(exc, "response", None)
            status = getattr(response, "status_code", None)
            if status not in (429, 500, 502, 503, 504):
                raise
            if attempt == attempts - 1:
                raise RuntimeError(f"HF read {label} still unavailable after {attempts} attempts; rerun this cell later.") from None
            headers = {k.lower(): v for k, v in getattr(response, "headers", {}).items()}
            delays = [min(300, 5 * 2**attempt)]
            hint = headers.get("retry-after", "")
            if hint:
                try:
                    delays.append(float(hint))
                except ValueError:
                    try:
                        delays.append(parsedate_to_datetime(hint).timestamp() - time.time())
                    except (ValueError, TypeError, OverflowError):
                        pass
            for match in re.finditer(r'\bt\s*=\s*(\d+)', headers.get("ratelimit", "")):
                delays.append(float(match.group(1)))
            body_delay = tl.parse_retry_after(str(exc))
            if body_delay is not None:
                delays.append(body_delay)
            remaining = max(delays) + 2
            print(f"[HF read] {label}: HTTP {status}; waiting {remaining:.0f}s, retry {attempt+1}/{attempts-1}.", flush=True)
            # Short sleeps allow Kaggle Stop to interrupt the wait normally.
            while remaining > 0:
                step = min(10, remaining)
                time.sleep(step)
                remaining -= step


import yaml
SOURCE_ROOT = Path(sess.stage_dir)/'s4b_source'
def source_file(rel, revision=SOURCE_REVISION):
    return Path(hf_read(rel,lambda:hf_hub_download(REPO,rel,repo_type='dataset',revision=revision,
        token=READ_TOKEN,local_dir=str(SOURCE_ROOT/revision))))
def source_csv(rel,revision=SOURCE_REVISION):
    return pd.read_csv(source_file(rel,revision))
selection=source_csv('tables/stage_b_selection.csv')
effects=source_csv('tables/stage_b_effects.csv')
PLAN=make_plan(selection,effects)
HASH=plan_hash(PLAN)
PREFIX=f'confirmations/{REVISION}/{HASH}'
OUT=Path(sess.stage_dir)/'s4b'/HASH
OUT.mkdir(parents=True,exist_ok=True)
cfgs=configs(sess,tl,PLAN)
run_ids=[cfg['run_id'] for cfg in cfgs]
base_ids=[f'a-{arch}-base-f1-s{seed}' for arch in PLAN['architectures'] for seed in PLAN['seeds']]
baseline=pd.concat([source_csv(f'runs/{rid}/metrics/final.csv') for rid in base_ids],ignore_index=True)
assert len(baseline)==6 and baseline.run_id.is_unique
assert baseline.status.eq('completed').all() and baseline.epochs_trained.eq(60).all()
assert baseline.epochs_planned.eq(60).all()
for rid in base_ids:
    old=yaml.safe_load(source_file(f'runs/{rid}/config.yaml').read_text())
    expected=sess.config(old['arch'],1,int(old['seed']),stage='a')
    differences={k:(old.get(k),expected[k]) for k in tl.RECIPE if k!='num_workers' and old.get(k)!=expected[k]}
    assert not differences, f'Baseline recipe changed: {rid} {differences}; do not train a mismatched comparison'
print('Frozen S4b plan:',PLAN)
print('New runs:',len(run_ids),'reused baselines:',len(base_ids))
print('No masks or new annotations required; fold 1 only.')


stage_b_selection.csv: 0.00B [00:00, ?B/s]

stage_b_effects.csv: 0.00B [00:00, ?B/s]

final.csv:   0%|          | 0.00/697 [00:00<?, ?B/s]

final.csv:   0%|          | 0.00/696 [00:00<?, ?B/s]

final.csv:   0%|          | 0.00/704 [00:00<?, ?B/s]

...

config.yaml:   0%|          | 0.00/514 [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/512 [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/512 [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/512 [00:00<?, ?B/s]

Frozen S4b plan: {'revision': 's4b-2026-09-09-r1', 'source_revision': 'bf62f9e9cbedacc580aa42542da14a068b8f9215', 'architectures': ['convnextv2_t', 'mobilenetv4'], 'discovery_architectures': ['regnety016', 'densenet121', 'resnet50'], 'factors': ['sampler_classweighted', 'transfer_random', 'sampler_uniform'], 'overrides': {'sampler_classweighted': {'sampler_name': 'class_weighted'}, 'transfer_random': {'pretrained': False}, 'sampler_uniform': {'sampler_name': 'uniform'}}, 'folds': [1], 'seeds': [1, 2, 3], 'epochs': 60, 'primary': 'best_val_f1_macro selected by best QWK, paired by architecture/fold/seed', 'secondary': 'final_val_f1_macro at epoch 60', 'ranking': 'descending mean signed paired NB06 selected-epoch F1 delta across all 9 discovery runs; alphabetical tie break', 'discovery_mean_delta': {'sampler_classweighted': 0.052668783256480674, 'transfer_random': -0.020624798760083045, 'sampler_uniform': -0.03344530971032074, 'head_ce': -0.04914985219660915, 'prep_clahe': -0.082673783753

In [6]:
SMOKE = "import sys,torch,tyrelib as tl\narch,bs,expected=sys.argv[1],int(sys.argv[2]),int(sys.argv[3])\nassert torch.cuda.device_count()>=2\nfmt=torch.channels_last\ntorch.manual_seed(4622)\nmodel=tl.build_model(arch,3,pretrained=False,head='coral',img_size=384)\nassert sum(p.numel() for p in model.parameters())==expected,'Architecture parameter count differs from published baseline'\nmodel=torch.nn.DataParallel(model.cuda().to(memory_format=fmt)).train()\nopt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=.05)\nscaler=tl._grad_scaler(torch.device('cuda'))\nx=torch.randn(bs,3,384,384,device='cuda').to(memory_format=fmt)\ny=torch.arange(bs,device='cuda')%3\nwith tl._autocast(torch.device('cuda')):\n    loss=tl.CoralHead.loss(model(x),y)\nassert torch.isfinite(loss)\nscaler.scale(loss).backward();scaler.unscale_(opt)\ntorch.nn.utils.clip_grad_norm_(model.parameters(),5.)\nscaler.step(opt);scaler.update();torch.cuda.synchronize()\nprint('PASS',arch,'batch',bs,'parameters',expected,'loss',float(loss.detach()),'torch',torch.__version__)\n"

In [7]:
import torch,sys,subprocess
assert torch.cuda.device_count()>=2, 'Select Kaggle GPU T4 x2'
assert all('T4' in torch.cuda.get_device_name(i) for i in (0,1)), 'This notebook is validated for dual T4; do not silently change the runtime'
# Publish the selection rule BEFORE claiming any experiment. Never use new results to re-rank.
plan_path=OUT/f'protocol_{ACCOUNT}.json'
plan_path.write_text(json.dumps(PLAN,indent=2))
sess.uploader.enqueue(plan_path,f'{PREFIX}/workers/{plan_path.name}')
assert sess.push_now('S4b frozen protocol'), 'HF publication failed; do not start unpublished experiments'
print(sess.reconcile(run_ids).to_string(index=False))
for arch in PLAN['architectures']:
    mine=[rid for rid in run_ids if f'-{arch}-' in rid and sess.inventory.state(rid)!='completed']
    if not mine:
        continue
    expected=int(baseline.loc[baseline.arch.eq(arch),'n_params_total'].iloc[0])
    p=subprocess.run([sys.executable,'-c',SMOKE,arch,str(tl.ZOO[arch]['bs']),str(expected)],
        capture_output=True,text=True,timeout=900,cwd=str(Path.cwd()))
    log=OUT/f'preflight_{arch}_{ACCOUNT}.txt'
    log.write_text(p.stdout+'\n'+p.stderr)
    sess.uploader.enqueue(log,f'{PREFIX}/workers/{log.name}')
    assert sess.push_now(f'S4b dual-T4 preflight {arch}'), 'Preflight upload failed'
    print(log.read_text())
    assert p.returncode==0, 'Preflight failed before claims; no model/batch reduction was made'
try:
    summaries=sess.run_all(cfgs,title='S4b two-architecture confirmation',isolate_runs=True,
        steal_stale=False,takeover_when_idle=True)
finally:
    sess.push_now('S4b training stopped or major cell complete')
assert sess.finish(), 'Pending HF upload; retry final flush before leaving'
sess.confirm_on_hf(run_ids)
print('When all 18 runs are FINISHED, run NB12R once for the joint report. One worker finishing is not all-team completion.')


[HF] flush (S4b frozen protocol): 1 file(s)
[HF] commit #1: 1 file(s), 0.0 MB, 0.8s  [1/25 this hr]
                                           run_id  state  epoch status_file best_qwk registry action
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s1 absent      0          NA       NA        -  train
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s2 absent      0          NA       NA        -  train
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s3 absent      0          NA       NA        -  train
      c-convnextv2_t-s4b_r1_sampler_uniform-f1-s1 absent      0          NA       NA        -  train
      c-convnextv2_t-s4b_r1_sampler_uniform-f1-s2 absent      0          NA       NA        -  train
      c-convnextv2_t-s4b_r1_sampler_uniform-f1-s3 absent      0          NA       NA        -  train
      c-convnextv2_t-s4b_r1_transfer_random-f1-s1 absent      0          NA       NA        -  train
      c-convnextv2_t-s4b_r1_transfer_random-f1-s2 absent      0          NA       NA        

acc2_w1_387dcf.jsonl: 0.00B [00:00, ?B/s]

acct1_w0_01e0de.jsonl: 0.00B [00:00, ?B/s]

acct1_w0_051f90.jsonl: 0.00B [00:00, ?B/s]

acct1_w0_053e1b.jsonl: 0.00B [00:00, ?B/s]

...

[ISOLATE] c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s2: child exited rc=0; status=paused epoch=26. Its process memory is now fully reclaimed.
[WATCHDOG] c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s2: checkpoint is safe at epoch 26; session is nearly over
[RUN] isolated child remained RAM-blocked at epoch 26; checkpoint is safe
[RUN] stopping worker after host_ram_guard. The checkpoint is on HuggingFace; use a fresh Kaggle session and re-run this notebook to resume at the next epoch.

                                           run_id         arch  fold  seed status  best_val_qwk  best_val_f1_macro  best_val_acc  epochs_trained  total_wall_seconds  total_energy_wh
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s2 convnextv2_t     1     2 paused      0.911894            0.62037       0.84375              26        27525.418013       792.746954
[SESSION] final flush -- blocking until HuggingFace confirms
[SESSION] done. commits=4 failures=0 pushed=0 MB


STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

                                           run_id       on_hf  epoch  missing_files
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s1   RESUMABLE     25              0
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s2   RESUMABLE     26              0
c-convnextv2_t-s4b_r1_sampler_classweighted-f1-s3   RESUMABLE     23              0
 c-mobilenetv4-s4b_r1_sampler_classweighted-f1-s1 NOT STARTED      0              4
 c-mobilenetv4-s4b_r1_sampler_classweighted-f1-s2 NOT STARTED      0              4
 c-mobilenetv4-s4b_r1_sampler_classweighted-f1-s3 NOT STARTED      0              4
      c-convnextv2_t-s4b_r1_transfer_random-f1-s1 NOT STARTED      0              4
      c-convnextv2_t-s4b_r1_transfer_random-f1-s2 NOT STARTED      0              4
      c-convnextv2_t-s4b_r1_transfer_random-f1-s3 NOT STARTED      0              4
       c-mobilenetv4-s4b_r1_transfer_random-f1-s1 NOT STARTED      0              4
       c-mobilenetv4-s4b_r1_transfer_random-f1-s2 NOT STARTED      0        